# 07 — public send

emailwerk's anonymous `sendTemplateMail`: the single GraphQL mutation a
browser may post to `/graphql` with no credentials at all. It restores the
mailpress v2 contact-form capability without reopening the origin, so what
these checks mostly judge is what the gate still **refuses** —
introspection, every guarded field, caller-chosen recipients, unbounded
volume — and that no second, hand-written entrance was built alongside it
(there is no REST route; see `docs/public-send.md` in the emailwerk
working copy).

Static checks read the emailwerk sources; the live half runs against the
DEPLOYED instance (`JAEN_EMAILWERK_URL`) **without credentials**, because
that is precisely the caller under test. Checks that would mail a real
person are gated behind `JAEN_ALLOW_LIVE_MAIL=1` and SKIP otherwise.

In [ ]:
import jaen_testkit as k
k.start_run('07-public-send')
print(k.CONFIG['repo_root'])

In [ ]:
import glob, json, os, re

EW = k.CONFIG['emailwerk_dir']
INDEX = os.path.join(EW, 'src', 'index.ts')


def root_fields(source, root):
    """Field name -> resolver source, for one root type of the `graphql` export.

    Brace counting rather than an import: importing src/index.ts would boot the
    HTTP server, the queue and the SMTP listeners.
    """
    start = source.find('export const graphql = {')
    if start < 0:
        return {}
    opening = re.compile(r'\n  %s: \{' % root).search(source, start)
    if not opening:
        return {}
    head = opening.end() - 1
    depth = 0
    tail = len(source)
    for pos in range(head, len(source)):
        ch = source[pos]
        if ch == '{':
            depth += 1
        elif ch == '}':
            depth -= 1
            if depth == 0:
                tail = pos
                break
    body = source[head + 1:tail]
    marks = [(m.start(), m.end(), m.group(1))
             for m in re.finditer(r'^    ([A-Za-z_]\w*): ', body, re.M)]
    fields = {}
    for index, (begin, after, name) in enumerate(marks):
        end = marks[index + 1][0] if index + 1 < len(marks) else len(body)
        fields[name] = body[after:end].strip()
    return fields


with k.section('no second entrance'):
    with k.check('no hand-written public REST route exists') as c:
        c.expect_true(not os.path.exists(os.path.join(EW, 'src', 'public', 'http.ts')),
                      'src/public/http.ts absent')
        r = k.sh(['grep', '-rn', 'registerPublicRoutes', 'src'], cwd=EW)
        c.expect_true(r.rc == 1 and not r.stdout.strip(),
                      'no registerPublicRoutes in src/ (grep rc=%s)' % r.rc)
        listing = sorted(os.listdir(os.path.join(EW, 'src', 'public')))
        c.expect_equal(listing,
                       ['rate-limit.test.ts', 'rate-limit.ts', 'send.test.ts', 'send.ts'],
                       'src/public holds the branch and its tests only')

    with k.check('isPublicPath was not widened') as c:
        adapter = k.read_text(os.path.join(EW, 'src', 'auth', 'adapter.ts'))
        if adapter is None:
            c.fail('src/auth/adapter.ts missing', abort=True)
        m = re.search(r'export function isPublicPath\([^)]*\)[^{]*\{(.*?)\n\}', adapter, re.S)
        if not m:
            c.fail('isPublicPath not found in src/auth/adapter.ts', abort=True)
        body = m.group(1)
        prefixes = sorted(re.findall(r'path\.startsWith\("([^"]+)"\)', body))
        c.expect_equal(prefixes,
                       ['/__pylon/', '/oauth/google/callback', '/sign/', '/signed/'],
                       'the path exemptions are unchanged')
        c.expect_not_contains(body, '/graphql', '/graphql is not a public path')

    with k.check('the anonymous allowlist is exactly sendTemplateMail') as c:
        gate = k.read_text(os.path.join(EW, 'src', 'auth', 'anonymous.ts'))
        if gate is None:
            c.fail('src/auth/anonymous.ts missing', abort=True)
        m = re.search(r'ANONYMOUS_OPERATIONS[^=]*=\s*\[([^\]]*)\]', gate)
        names = re.findall(r'"([^"]+)"', m.group(1)) if m else []
        c.expect_equal(names, ['sendTemplateMail'], 'one allowlisted root field')
        c.expect_contains(gate, 'MAX_ANONYMOUS_BODY_BYTES', 'the body cap exists')
        c.expect_contains(gate, 'GRAPHQL_PATH', 'the gate is bound to one path')


with k.section('the guarded surface'):
    source = k.read_text(INDEX, '')
    query = root_fields(source, 'Query')
    mutation = root_fields(source, 'Mutation')

    with k.check('every other root field is still auth.guard-ed') as c:
        c.expect_true(len(query) > 0 and len(mutation) > 0,
                      '%d Query / %d Mutation root fields' % (len(query), len(mutation)))
        allowed = ['sendTemplateMail']
        unguarded = sorted(name for name, body in list(query.items()) + list(mutation.items())
                           if name not in allowed and not body.startswith('auth.guard('))
        c.expect_equal(unguarded, [], 'nothing unguarded besides the dual-mode mutation')
        if unguarded:
            c.detail('\n'.join(unguarded))
        c.expect_true('sendTemplateMail' not in query,
                      'the allowlisted name is a Mutation, never a Query')

    with k.check('the dual-mode resolver branches on the gate') as c:
        body = mutation.get('sendTemplateMail', '')
        c.expect_true(bool(body), 'Mutation.sendTemplateMail found')
        c.expect_contains(body, 'isAnonymousRequestContext', 'consults the gate')
        c.expect_contains(body, 'anonymousSendTemplateMail', 'routes through src/public/send.ts')
        c.expect_contains(source,
                          'const sendTemplateMailAsPrincipal = auth.guard(',
                          'the authenticated branch stays role-guarded')

    with k.check('the isPublic column and its migration exist') as c:
        schema = k.read_text(os.path.join(EW, 'prisma', 'schema.prisma'), '')
        c.expect_contains(schema, 'isPublic', 'the Prisma model carries the flag')
        hits = []
        for path in sorted(glob.glob(os.path.join(EW, 'prisma', 'migrations', '*', 'migration.sql'))):
            sql = k.read_text(path, '')
            if 'isPublic' in sql:
                hits.append((path, sql))
        c.expect_true(bool(hits), '%d migration(s) touch isPublic' % len(hits))
        if hits:
            c.note(os.path.basename(os.path.dirname(hits[0][0])))
            c.expect_contains(hits[0][1], 'ADD COLUMN', 'the column is added, not assumed')

    with k.check('an admin can flag an existing template public') as c:
        m = re.search(r'type TemplateUpdateArgs = \{(.*?)\n\};', source, re.S)
        if not m:
            c.fail('TemplateUpdateArgs not found in src/index.ts', abort=True)
        if 'isPublic' in m.group(1):
            c.ok('templateUpdate exposes isPublic')
        else:
            # The service layer takes it (src/services/template.ts), the GraphQL
            # args type does not, so the flag is reachable on create only.
            c.warn('templateUpdate does not expose isPublic: an EXISTING template '
                   'cannot be flagged public through the API')
            c.detail('docs/public-send.md "Operating it" step 1 names templateUpdate, '
                     'but TemplateUpdateArgs in src/index.ts has no isPublic field. '
                     'TemplateUpdateInput in src/services/template.ts does.')

In [ ]:
with k.section('emailwerk unit suites'):
    with k.check('the gate and both auth adapters') as c:
        r = k.sh('npx vitest run src/auth/anonymous.test.ts src/auth/dev.test.ts '
                 'src/auth/cfaccess.test.ts', cwd=EW, timeout=600)
        if r.rc == 127:
            c.skip('npx/vitest unavailable')
        c.require(r, 'vitest auth gate')
        c.expect_contains(r.stdout + r.stderr, 'Test Files  3 passed')

    with k.check('the public branch, its rate limit and the schema pin') as c:
        r = k.sh('npx vitest run src/public/send.test.ts src/public/rate-limit.test.ts '
                 'src/send/pipeline.test.ts src/schema.test.ts', cwd=EW, timeout=600)
        if r.rc == 127:
            c.skip('npx/vitest unavailable')
        c.require(r, 'vitest public send')
        c.expect_contains(r.stdout + r.stderr, 'Test Files  4 passed')

## Deployed instance

The live checks POST to the DEPLOYED emailwerk (`JAEN_EMAILWERK_URL`,
default `https://emailwerk.com/graphql`) **with no credentials**: an
anonymous caller is the subject of the test, not an obstacle to it. Admin
credentials are used only where a check needs to look up a template
(`JAEN_EMAILWERK_BASIC`, else the local ansible-vault); that half SKIPs
without them.

Bogus template ids never enqueue anything, so the rate-limit check hammers
with those and nothing is ever delivered. It runs last, because a tripped
per-IP window (5 in 10 minutes) would otherwise starve the checks after
it.

Three human steps, never run from this notebook:

```bash
# 1. deploy the build that carries the gate
cd ~/git/emailwerk-fido-test/deploy/k8s/emailwerk && ./redeploy.sh
```

```graphql
# 2. flag the contact template publicly sendable (admin credentials)
mutation { templateUpdate(id: "<template id>", isPublic: true) { id isPublic } }
```

```bash
# 3. only when a REAL mail to the template's stored recipients is wanted
JAEN_ALLOW_LIVE_MAIL=1 papermill 07-public-send.ipynb out/07-public-send-output.ipynb
```

In [ ]:
with k.section('anonymous, no credentials'):
    url = k.CONFIG['emailwerk_url']
    auth = k.emailwerk_auth_headers()
    BOGUS_ID = 'jaen-test-no-such-template-0000'
    gate_live = False
    bogus_message = ''
    contact_id = os.environ.get('JAEN_EMAILWERK_CONTACT_TEMPLATE', '')

    def gql_literal(value):
        """Render a Python value as a GraphQL argument literal."""
        if isinstance(value, dict):
            return '{%s}' % ', '.join('%s: %s' % (key, gql_literal(inner))
                                      for key, inner in value.items())
        if isinstance(value, list):
            return '[%s]' % ', '.join(gql_literal(item) for item in value)
        return json.dumps(value)

    def anon_send(args):
        """POST sendTemplateMail with NO Authorization header at all."""
        return k.graphql(url, 'mutation Contact { sendTemplateMail(args: %s) '
                              '{ id status } }' % gql_literal(args))

    def errors_of(result):
        payload = result.json() or {}
        return [str(e.get('message', '')) for e in (payload.get('errors') or [])]

    def codes_of(result):
        payload = result.json() or {}
        return [str((e.get('extensions') or {}).get('code', ''))
                for e in (payload.get('errors') or [])]

    def admin_templates(with_is_public=True):
        """Admin template list; falls back to the pre-isPublic shape."""
        fields = 'id description isPublic envelope { to }' if with_is_public \
            else 'id description envelope { to }'
        r = k.graphql(url, '{ templates { nodes { %s } } }' % fields, headers=auth)
        nodes = (((r.json() or {}).get('data') or {}).get('templates') or {}).get('nodes')
        return r, nodes or []

    with k.check('the gate does not open introspection') as c:
        if not url:
            c.skip('JAEN_EMAILWERK_URL not set')
        r = k.graphql(url, '{ __schema { queryType { name } } }')
        if r.error:
            c.skip('endpoint not reachable: %s' % k.preview(r.error, 60))
        c.expect_true(r.status in (401, 403), 'HTTP %s without credentials' % r.status)
        c.expect_true(not (((r.json() or {}).get('data') or {}).get('__schema')),
                      'no schema handed to an anonymous caller')

    with k.check('a guarded field stays closed to anonymous callers') as c:
        if not url:
            c.skip('JAEN_EMAILWERK_URL not set')
        r = k.graphql(url, '{ templates { totalCount } }')
        if r.error:
            c.skip('endpoint not reachable: %s' % k.preview(r.error, 60))
        c.expect_true(r.status in (401, 403), 'HTTP %s without credentials' % r.status)
        c.expect_true(not (((r.json() or {}).get('data') or {}).get('templates')),
                      'Query.templates returned no data')

    with k.check('the deployed build carries the anonymous gate') as c:
        if not url:
            c.skip('JAEN_EMAILWERK_URL not set')
        probe = anon_send({'templateId': BOGUS_ID})
        if probe.error:
            c.skip('endpoint not reachable: %s' % k.preview(probe.error, 60))
        if probe.status in (401, 403) and not (probe.json() or {}).get('errors'):
            c.skip('HTTP %s on the allowlisted mutation: the deployed build predates '
                   'the gate, redeploy pending' % probe.status)
        gate_live = True
        bogus_message = ' '.join(errors_of(probe))
        c.expect_true(bool(bogus_message) or probe.ok,
                      'the mutation reached the resolver (HTTP %s)' % probe.status)
        c.note(k.preview(bogus_message or probe.body, 80))

    with k.check('an unknown template id yields the uniform not-found error') as c:
        if not gate_live:
            c.skip('anonymous gate not deployed')
        text = bogus_message or ' '.join(errors_of(anon_send({'templateId': BOGUS_ID})))
        c.expect_true(bool(text), 'an error was returned, not a send')
        c.expect_contains(text, 'Vorlage nicht gefunden', 'the uniform message')
        for leak in ('isPublic', 'öffentlich', 'public', 'nicht freigegeben'):
            c.expect_not_contains(text, leak, 'no "not public" hint: %s' % leak)

    with k.check('a real private template is indistinguishable from a missing one') as c:
        if not gate_live:
            c.skip('anonymous gate not deployed')
        if not auth:
            c.skip('no admin credentials: cannot pick a real private template')
        listing, nodes = admin_templates()
        if not nodes:
            c.skip('admin templates query gave nothing: %s' % k.preview(listing.body, 60))
        private = [n for n in nodes if n.get('isPublic') is False]
        if not private:
            c.skip('no private template in the deployed org')
        r = anon_send({'templateId': private[0]['id']})
        c.note('%d private template(s), probing %s' % (len(private), private[0]['id']))
        c.expect_equal(' '.join(errors_of(r)), bogus_message,
                       'byte-identical to the bogus-id error')

    with k.check('an anonymous send cannot choose the recipients (to)') as c:
        if not gate_live:
            c.skip('anonymous gate not deployed')
        r = anon_send({'templateId': BOGUS_ID, 'to': ['jaen-test@example.com']})
        text = ' '.join(errors_of(r))
        codes = ' '.join(codes_of(r))
        c.expect_true(bool(text), 'HTTP %s, rejected' % r.status)
        c.expect_true('RECIPIENT_NOT_ALLOWED' in codes or 'keine Empfänger setzen' in text,
                      k.preview(text or codes, 80))
        c.expect_not_contains(text, 'Vorlage nicht gefunden',
                              'rejected before the template was even looked up')

    with k.check('an anonymous send cannot choose the recipients (envelopeOverride.to)') as c:
        if not gate_live:
            c.skip('anonymous gate not deployed')
        r = anon_send({'templateId': BOGUS_ID,
                       'envelopeOverride': {'to': 'jaen-test@example.com'}})
        text = ' '.join(errors_of(r))
        codes = ' '.join(codes_of(r))
        c.expect_true(bool(text), 'HTTP %s, rejected' % r.status)
        c.expect_true('RECIPIENT_NOT_ALLOWED' in codes or 'keine Empfänger setzen' in text,
                      k.preview(text or codes, 80))

    with k.check('the netsnek contact template is flagged public') as c:
        if not url:
            c.skip('JAEN_EMAILWERK_URL not set')
        if not auth:
            c.skip('no admin credentials (JAEN_EMAILWERK_BASIC or vault)')
        listing, nodes = admin_templates()
        has_flag = bool(nodes)
        if not nodes:
            listing, nodes = admin_templates(with_is_public=False)
        if not nodes:
            c.skip('admin templates query gave nothing: %s' % k.preview(listing.body, 60))

        def contact_rank(node):
            """How well a template matches "the netsnek contact form"; 0 = not."""
            description = (node.get('description') or '').lower()
            recipients = ' '.join((node.get('envelope') or {}).get('to') or []).lower()
            if 'contact' not in description or 'confirmation' in description:
                return 0
            if 'office@netsnek.com' in recipients:
                return 2
            return 1 if 'netsnek.com' in recipients else 0

        if contact_id:
            found = next((n for n in nodes if n['id'] == contact_id), None)
        else:
            ranked = sorted(((contact_rank(n), n) for n in nodes),
                            key=lambda pair: pair[0], reverse=True)
            found = ranked[0][1] if ranked and ranked[0][0] > 0 else None
            contact_id = found['id'] if found else ''
        if not found:
            c.skip('no netsnek contact template among %d templates' % len(nodes))
        c.note('%s "%s" -> %s' % (found['id'], found.get('description'),
                                  ', '.join((found.get('envelope') or {}).get('to') or [])))
        mutation_text = ('mutation { templateUpdate(id: "%s", isPublic: true) '
                         '{ id isPublic } }' % found['id'])
        if not has_flag:
            c.warn('the deployed schema has no TemplateView.isPublic yet, so nothing '
                   'is public: it ships with the redeploy')
            c.detail('then, as admin:\n' + mutation_text)
        elif found.get('isPublic'):
            c.ok('isPublic is set')
        else:
            c.warn('not flagged public yet')
            c.detail('as admin:\n' + mutation_text)

    with k.check('a real anonymous send is accepted') as c:
        if not gate_live:
            c.skip('anonymous gate not deployed')
        if os.environ.get('JAEN_ALLOW_LIVE_MAIL') != '1':
            c.skip('JAEN_ALLOW_LIVE_MAIL is not 1: refusing to send a real mail')
        if not contact_id:
            c.skip('no public contact template to send through')
        r = anon_send({
            'templateId': contact_id,
            'values': {'name': 'jaen test suite',
                       'email': 'office@netsnek.com',
                       'message': 'Automated check from tests/07-public-send.ipynb.'},
            'envelopeOverride': {'replyTo': os.environ.get('JAEN_LIVE_MAIL_REPLY_TO',
                                                           'office@netsnek.com')},
        })
        sent = ((r.json() or {}).get('data') or {}).get('sendTemplateMail') or {}
        c.expect_true(r.ok and not errors_of(r),
                      'HTTP %s %s' % (r.status, k.preview(r.body, 60)))
        c.expect_true(bool(sent.get('id')), 'message id %s' % sent.get('id'))
        c.note('status %s' % sent.get('status'))

    # Last: a tripped per-IP window (5 per 10 min) would starve every check
    # above. Bogus ids never enqueue, so this delivers nothing.
    with k.check('repeated anonymous sends run into the rate limit') as c:
        if not gate_live:
            c.skip('anonymous gate not deployed')
        limited_at = None
        for attempt in range(1, 13):
            r = anon_send({'templateId': BOGUS_ID})
            text = ' '.join(errors_of(r))
            codes = ' '.join(codes_of(r))
            if r.status == 429 or 'RATE_LIMITED' in codes or 'Zu viele Anfragen' in text:
                limited_at = attempt
                c.note(k.preview(text or codes, 60))
                break
        c.expect_true(limited_at is not None,
                      'rate limited after %s bogus-id sends' % limited_at)

In [ ]:
k.summary()
k.save_results('results-07-public-send.json')
rc = k.verdict()
assert rc == 0, 'run has FAILures — see the summary above'